# Day 19 — Query Optimization Mindset
**Estimated time:** 60–75 min
**Datasets:** `sales_orders.csv`, all tables

## Learning Objectives
- Read and interpret `EXPLAIN QUERY PLAN` output in SQLite
- Understand when SQLite uses indexes vs full scans
- Recognize sargable vs non-sargable predicates — write queries that can use indexes

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = "../../data/raw"

# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- EXPLAIN QUERY PLAN: SQLite's optimizer output --
# Tells you whether the engine scans the whole table or uses an index
import sqlite3 as sqlite3_raw

def explain(sql):
    cur = con.execute(f"EXPLAIN QUERY PLAN {sql}")
    rows = cur.fetchall()
    for r in rows:
        print(r)
    print()

# Baseline: full table scan
print("=== FULL TABLE SCAN ===")
explain("SELECT * FROM SALES WHERE NETWR > 10000")

In [ ]:
# -- Create an index and compare --
con.execute("CREATE INDEX IF NOT EXISTS idx_sales_netwr ON SALES(NETWR)")
con.execute("CREATE INDEX IF NOT EXISTS idx_sales_erdat ON SALES(ERDAT)")
con.execute("CREATE INDEX IF NOT EXISTS idx_sales_kunnr ON SALES(KUNNR)")

print("=== AFTER INDEX ON NETWR ===")
explain("SELECT * FROM SALES WHERE NETWR > 10000")

print("=== INDEX on ERDAT ===")
explain("SELECT * FROM SALES WHERE ERDAT >= '2024-01-01'")

In [ ]:
# -- Sargable vs Non-Sargable predicates --
# SARGABLE = can use an index. Non-sargable = forces full scan.
#
# NON-SARGABLE (wraps column in function -> index unusable):
print("=== NON-SARGABLE: function on indexed column ===")
explain("SELECT * FROM SALES WHERE strftime('%Y', ERDAT) = '2024'")

# SARGABLE (range on the column directly):
print("=== SARGABLE: direct range comparison ===")
explain("SELECT * FROM SALES WHERE ERDAT >= '2024-01-01' AND ERDAT < '2025-01-01'")

print("Key rule: never apply a function to an indexed column in WHERE.")
print("Transform your constant instead: strftime('%Y', col) = '2024' -> col >= '2024-01-01' AND col < '2025-01-01'")

In [ ]:
# -- SELECT * vs explicit columns --
# SELECT * fetches all columns; explicit selection reduces I/O
# On large tables this matters significantly

print("=== SELECT * ===")
explain("SELECT * FROM SALES WHERE KUNNR = 'CUST001'")

print("=== Explicit columns (covering index opportunity) ===")
explain("SELECT VBELN, NETWR FROM SALES WHERE KUNNR = 'CUST001'")

In [ ]:
# -- Rewriting patterns: avoid OR on indexed columns --
# OR prevents SQLite from using a single index efficiently
# Use UNION ALL instead for better plan

print("=== OR condition ===")
explain("SELECT * FROM SALES WHERE STATUS = 'OPEN' OR STATUS = 'PENDING'")

print("=== UNION ALL equivalent (can use index for each branch) ===")
explain(
    """
    SELECT * FROM SALES WHERE STATUS = 'OPEN'
    UNION ALL
    SELECT * FROM SALES WHERE STATUS = 'PENDING'
    """
)

In [ ]:
# -- Correlated subquery vs JOIN: performance comparison --
# Correlated subqueries re-execute for every row -- avoid at scale

# Method 1: Correlated subquery (slower)
import time
t0 = time.perf_counter()
result1 = q(
    """
    SELECT KUNNR, NETWR,
           (SELECT SUM(NETWR) FROM SALES inner_s WHERE inner_s.KUNNR = outer_s.KUNNR) AS customer_total
    FROM SALES outer_s
    LIMIT 500
    """
)
t_corr = time.perf_counter() - t0

# Method 2: CTE aggregation + JOIN (faster)
t0 = time.perf_counter()
result2 = q(
    """
    WITH customer_totals AS (
        SELECT KUNNR, SUM(NETWR) AS customer_total FROM SALES GROUP BY KUNNR
    )
    SELECT s.KUNNR, s.NETWR, ct.customer_total
    FROM SALES s
    JOIN customer_totals ct ON s.KUNNR = ct.KUNNR
    LIMIT 500
    """
)
t_join = time.perf_counter() - t0

print(f"Correlated subquery: {t_corr*1000:.1f}ms")
print(f"CTE + JOIN:          {t_join*1000:.1f}ms")
print(f"Speedup: {t_corr/t_join:.1f}x")

---
## Daily Prompt

**Take the chained CTE query from Day 15 and rewrite it two ways:**
1. With nested subqueries (the before version)
2. With CTEs (the after version from day 15)

Run EXPLAIN QUERY PLAN on both and compare the output.

    # YOUR SOLUTION
    nested_sql = '...your nested subquery version...'
    cte_sql    = '...your CTE version from day 15...'

    print('=== Nested subquery plan ===')
    explain(nested_sql)
    print('=== CTE version plan ===')
    explain(cte_sql)

    import time
    t0 = time.perf_counter(); q(nested_sql); t_nested = time.perf_counter() - t0
    t0 = time.perf_counter(); q(cte_sql);   t_cte    = time.perf_counter() - t0
    print(f'Nested: {t_nested*1000:.1f}ms | CTE: {t_cte*1000:.1f}ms')

**Key insight:** SQLite treats CTEs as materialized temp tables. The optimizer may produce identical plans for equivalent logic, but CTEs are almost always preferable for readability and maintainability — which matters more than micro-optimizations for analytics workloads.